In [2]:
import json
import os
import pandas as pd

# paths
BASE_DIR = "/home/aparnabg/orcd/scratch/bidsdata"
BIDS_ROOT = "/home/aparnabg/orcd/scratch/bidsdata/final_bids-dataset"

def merge_logs():
    all_processed, all_failed = [], []
    task_dirs = sorted(
        int(item) for item in os.listdir(BASE_DIR)
        if item.isdigit() and os.path.isdir(os.path.join(BASE_DIR, item))
    )
    
    for task_id in task_dirs:
        task_dir = os.path.join(BASE_DIR, str(task_id))
        processed_file = os.path.join(task_dir, "processing_log.json")
        failed_file = os.path.join(task_dir, "not_processed.json")

        if os.path.exists(processed_file):
            with open(processed_file, 'r') as f:
                all_processed.extend(json.load(f))
        
        if os.path.exists(failed_file):
            with open(failed_file, 'r') as f:
                all_failed.extend(json.load(f))
    
    return all_processed, all_failed

processed_data, failed_data = merge_logs()

# Save all logs
with open(os.path.join(BASE_DIR, "all_processing_log.json"), 'w') as f:
    json.dump(processed_data, f, indent=2, default=str)

with open(os.path.join(BASE_DIR, "all_not_processed.json"), 'w') as f:
    json.dump(failed_data, f, indent=2, default=str)

# Create participants files
if processed_data:
    participant_ids = {
        entry['participant_id'] for entry in processed_data if 'participant_id' in entry
    }

    participants_data = [
        {"participant_id": f"sub-{pid}", "age": "n/a", "validity": "n/a"}
        for pid in sorted(participant_ids)
    ]
    
    pd.DataFrame(participants_data).to_csv(
        os.path.join(BIDS_ROOT, "participants.tsv"), 
        sep='\t', index=False, na_rep='n/a'
    )
    
    participants_json = {
        "participant_id": {"Description": "Unique participant identifier"},
        "age": {"Description": "Age information", "Units": "months"}, 
        "validity": {"Description": "Data validity information"}
    }
    
    with open(os.path.join(BIDS_ROOT, "participants.json"), 'w') as f:
        json.dump(participants_json, f, indent=2)

# Summary
print(f"Total processed: {len(processed_data)}")
print(f"Total failed: {len(failed_data)}")

if processed_data:
    task_counts = {}
    for entry in processed_data:
        task = entry.get('task_label', 'unknown')
        task_counts[task] = task_counts.get(task, 0) + 1
    
    print("\nTasks processed:")
    for task, count in sorted(task_counts.items()):
        print(f"  {task}: {count}")
    
    session_counts = {}
    for entry in processed_data:
        session = entry.get('session_id', 'unknown')
        session_counts[session] = session_counts.get(session, 0) + 1
    
    print("\nSessions:")
    for session, count in sorted(session_counts.items()):
        print(f"  Session {session}: {count}")

if failed_data:
    error_counts = {}
    for entry in failed_data:
        error = entry.get('error', 'Unknown')
        error_counts[error] = error_counts.get(error, 0) + 1
    
    print("\nFailed videos by error type:")
    for error, count in sorted(error_counts.items()):
        print(f"  {error}: {count}")


Total processed: 3259
Total failed: 20

Tasks processed:
  bookshare: 56
  dailyroutine: 252
  generalsocialcommunicationinteraction: 1010
  generalsocialinteraction: 1
  motorplay: 497
  other: 283
  socialroutine: 158
  specialoccasion: 135
  toyplay: 564
  unknown: 303

Sessions:
  Session 01: 1677
  Session 02: 1582

Failed videos by error type:
  Audio extraction failed: ffmpeg version N-121159-g0bd5a7d371-20250921 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (crosstool-NG 1.28.0.1_403899e)
  configuration: --prefix=/ffbuild/prefix --pkg-config-flags=--static --pkg-config=pkg-config --cross-prefix=x86_64-ffbuild-linux-gnu- --arch=x86_64 --target-os=linux --enable-gpl --enable-version3 --disable-debug --enable-iconv --enable-zlib --enable-libxml2 --enable-libsoxr --enable-openssl --enable-libvmaf --enable-fontconfig --enable-libharfbuzz --enable-libfreetype --enable-libfribidi --enable-vulkan --enable-libshaderc --enable-libvorbis --enable-libxcb --enable-x